In [ ]:
%pylab inline
import scipy as sc
from scipy import optimize
from copy import copy
import eucare as ec

In [ ]:
def kawasaki_sum(v):
    angles = np.abs(np.array([e['in_angle'] for e in v.incoming_iter()]))
    assert len(angles) % 2 == 0
    return np.sum(angles * (-1) ** np.arange(len(angles)))
    
def max_kawasaki_sum(vertices):
    if isinstance(vertices, ec.half.HalfEdgeGraph):
        vertices = [v for v in vertices.vertices if not v.on_border()]
    return np.max([kawasaki_sum(v) for v in vertices])

In [ ]:
render_settings = dict(face_inset=0, render_vertices=False, render_faces=False)
# render_settings = dict(
#     width=2000,
#     height=2000,
#     figsize=(8, 8),
#     #scale=100,
#     render_edges=True,
#     render_faces=True,
#     render_vertices=False,
#     face_inset=0,
#     for_cutting = False,
#     line_width=7
# )

In [ ]:
#TODO: use incircle center for dual graph generation. will give more symmetric results

def delete_rings_around_face(G, face_to_delete, n_rings):
    current_ring = [face_to_delete] if isinstance(face_to_delete, ec.half.Face) else face_to_delete
    for _ in range(n_rings-1):
        next_ring = set(f2 
                        for f1 in current_ring 
                        for v in f1.vertex_iter()
                        for f2 in v.face_iter()
                        if f2 in G.faces and f2 not in current_ring)
        [G.delete_face(f) for f in current_ring]
        current_ring = next_ring
    [G.delete_face(f) for f in current_ring]
        
def doyle_graph(n=25, rings=21, factor=0.2, deletion_rings=5):
    
    G = ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
    G = ec.half.EuclideanPositionHEG(other=G)
    
    for i in range(rings):
        G = ec.conway.loft_graph(t=factor)(
            G, 
            delete_on_border=False,
            faces=[f for f in G.faces if f.order() == n]
        )
        
    ec.colorization.congruency_colorize(G)
    
    # get the faces to delete and delete them
    central_ngon = next(iter(f for f in G.faces if f.order() == n))
    central_right_edge = next(iter(e for e in central_ngon.halfedge_iter() 
                                   if e.orig['pos'][1] < 0 and e.dest['pos'][1] > 0))
    
    # add the outer face
    outer_face = ec.half.Face(any_side=G.get_any_border())
    G.faces.add(outer_face)
    for e in G.border_edge_iter():
        e.face = outer_face
    G.check_consistency()
    
    def one_ring_outwards(e):
        return e.rev.nex.nex
    right_edge = central_right_edge
    for _ in range((rings+1)//2):
        right_edge = one_ring_outwards(right_edge)
    face_to_delete = right_edge.face
    initial_ring = [face_to_delete, 
                   right_edge.nex.rev.face,
                   right_edge.pre.rev.face]
    singularity = complex(((right_edge.orig['pos'][0]**2 + right_edge.nex.nex.dest['pos'][0]**2)**0.5)*(1-factor), 0)
    print(singularity)
    
    delete_rings_around_face(G, initial_ring, deletion_rings)
    
    G.show(**render_settings)
    ps, vs = G.get_position_view()
    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    
    k = k / (k - singularity)
    
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    
    ps[:] = k
    G.recompute_lengths_and_angles()
    return G

#G = doyle_graph()
#G.show(**render_settings)
#G.check_consistency()

In [ ]:
from eucare.classifiers import *
import eucare as eu
from eucare.example_tilesets import *
from eucare.example_graphs import *
from eucare import conway
from copy import deepcopy

def test_graph():
    G = from_tiles(eu.example_tilesets.t_3_3_4_3_4(), rings=3)
    #G = from_tiles(eu.example_tilesets.platonic(6), rings=3)

    plotting_kwargs = {
        'figsize': (10, 10),
        'render_faces': True,
        'render_vertices': False,
        'render_edges': False,
        'face_inset': 0,
    }
    def show():
        global G
        cc = congruency_classifier()

        for f in G.faces:
            f['color_key'] = cc.classify(f)

        G.show(**plotting_kwargs)

    show()


    G = conway.gyro_graph()(G)
    G = conway.dual_graph()(G)
    return G


In [ ]:
def fancy_graph():
    #return doyle_graph()
#     hextile0 = ec.prototiles.RegularEuclideanTile(6, edge_labels=[0]*6)
#     hextile1 = ec.prototiles.RegularEuclideanTile(6, edge_labels=[0]*6)
#     tritile = ec.prototiles.RegularEuclideanTile(3, edge_labels=[0]*3)
#     ec.example_tilesets.align_tiles(hextile0, 0, hextile1, 0)
#     ec.example_tilesets.align_tiles(hextile1, 0, tritile, 0)
#     ec.example_tilesets.align_tiles(tritile, 0, tritile, 0)

     #G = ec.example_graphs.from_tiles([tritile, hextile1], 4)
#     G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(6), 8)
#     central_face = next(iter(f for f in G.faces if np.linalg.norm(f.midpoint()) < 1e-6))
#     delete_rings_around_face(G, central_face, 4)
#     G = ec.conway.ambo_graph()(G, delete_on_border=True)
    
#     # u2 Graph with nice borders
#     G = ec.example_graphs.from_tiles(ec.example_tilesets.u2_4_6_12__3_4_6_4(), 3, base_tile=-1, vertex_based=False)
#     for e in G.border_edges():
#         if e.rev.face.order() == 4 and e.nex.rev.face.order() == 4:
#             G.execute_edge_instruction(e)
#             G.execute_edge_instruction(e.rev.rev.pre.rev)
#     # optionally, merge triangles
#     G = ec.conway.join_graph()(G, faces=[f for f in G.faces if f.order()==3])
    
    # 3.12.12
    G = ec.example_graphs.from_tiles(ec.example_tilesets.t_3_12_12(), 1)
    to_expand_from = [e for e in G.border_edges()
                      if e in G.halfedges and e.on_border() and e.dest.order() == 3]
    for e in to_expand_from:
        G.execute_edge_instruction(e)

    #G = ec.example_graphs.from_tiles(ec.example_tilesets.t_4_6_12(), 3)
            
    #G = ec.example_graphs.rosette(11)
    #G.show(**render_settings)
    
    #G = ec.conway.chamfer_graph(t=1/3)(G, faces=[f for f in G.faces if f.order() == 12 and np.linalg.norm(f.midpoint()) > 0.1])
    
    ## just an n-gon
    
    n = 12
    r = 3
    #G = ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
    #G = ec.half.EuclideanPositionHEG(other=G)
    # Idea: first chamfer without border-delete, then loft.
    factor = 2.8
#     G = ec.conway.chamfer_graph(t=1/(factor**1.6))(
#         G, 
#         delete_on_border=False,
#         delete_inner_border=False,
#         faces=[f for f in G.faces if f.order() == n]
#     )
    G = ec.conway.chamfer_graph(t=1/(factor))(
        G, 
        delete_on_border=False,
        delete_inner_border=False,
        faces=[f for f in G.faces if f.order() == n]
    )
    for i in range(r):
        G = ec.conway.loft_graph(t=1/factor)(
            G, 
            delete_on_border=False,
            faces=[f for f in G.faces if f.order() == n]
        )
    
    
    #G = ec.conway.kis_graph()(G)
    #G = ec.conway.dual_graph()(G)
    #ec.colorization.congruency_colorize(G)
    #G.show(**render_settings)

    ps, vs = G.get_position_view()

    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    
    #k = 1/k
    #k -= complex(*platonic(3)[0].points[-1])
    #k = k ** 2
    #k = np.exp(k*np.pi/5/5)
    
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    #k[:, 0] *= 0.5

    ps[:] = k
    G.recompute_lengths_and_angles()
    return G

G = fancy_graph()
G.show(**render_settings)
G.check_consistency()

In [ ]:
def concentric_rings(n, rings, factor):
    assert factor > 1
    r = rings
    G = ec.prototiles.RegularEuclideanTile(n).make_graph(add_positions=True)[0]
    G = ec.half.EuclideanPositionHEG(other=G)
    # Idea: first chamfer without border-delete, then loft.
    G = ec.conway.chamfer_graph(t=1/factor)(
        G, 
        delete_on_border=False,
        delete_inner_border=False,
        faces=[f for f in G.faces if f.order() == n]
    )
    for i in range(r):
        G = ec.conway.loft_graph(t=1/factor)(
            G, 
            delete_on_border=False,
            faces=[f for f in G.faces if f.order() == n]
        )
    ps, vs = G.get_position_view()
    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    ps[:] = k
    G.recompute_lengths_and_angles()
    return G


def archimedian_4_6_12(rings=3, smooth_border=False):
    G = ec.example_graphs.from_tiles(ec.example_tilesets.t_4_6_12(), rings)
    to_join = []
    
    if smooth_border:
        for v in G.vertices:
            if v.on_border() and v.order() == 2:
                to_join.append(v)
        for v in to_join:
            G.join_vertex(v)
        
        
    ps, vs = G.get_position_view()

    k = ps.copy()
    k = np.array([complex(*ki) for ki in k])
    
    #k = 1/k
    #k -= complex(*platonic(3)[0].points[-1])
    #k = k ** 2
    #k = np.exp(k*np.pi/5/5)
    
    k -= np.mean(k)
    k = k / np.max(np.abs(k)) * 3
    k = np.stack([k.real, k.imag], axis=-1)
    ps[:] = k
    
    G.recompute_lengths_and_angles()
    return G


from eucare.overlap import THIS_WAY, assign_shrink_rotate_creases

def is_undirected(e):
    return not (THIS_WAY in e.attributes or THIS_WAY in e.rev.attributes)

G = archimedian_4_6_12(3)
#G = ec.conway.chamfer_graph()(G)
#G = ec.conway.dual_graph()(G)
#G = concentric_rings(12, 3, 2.5)
#G = concentric_rings(12, 4, 4.5)
#G = ec.conway.kis_graph()(G)#, faces=[f for f in G.faces if f.order() == 4])
#G = ec.conway.kis_graph()(G)
#G = ec.example_graphs.from_tiles(ec.example_tilesets.platonic(6), 2)
#G = ec.conway.join_graph()(G)
#G = ec.conway.gyro_graph()(G)
#G = ec.conway.kis_graph()(G)
G = ec.conway.kis_graph()(G, 
                          delete_on_border=False,
                          faces=[f for f in G.faces if f.order() == 6])
r = 1
G = ec.conway.chamfer_graph(t=1/(2.8))(
        G, 
        delete_on_border=False,
        delete_inner_border=False,
        faces=[f for f in G.faces if f.order() == 12]
    )
for i in range(r):
    G = ec.conway.loft_graph(t=1/2.8)(
        G, 
        delete_on_border=False,
        faces=[f for f in G.faces if f.order() == 12]
    )
    
        
for f in G.faces:
    if f.order() == 12:
        for e in f.halfedge_iter():
            e[THIS_WAY] = True
            for _ in range(r+1):
                e = e.rev.nex.nex
                e.pre[THIS_WAY] = True
                e[THIS_WAY] = True
                
#G = ec.conway.dual_graph()(G)
#G = ec.conway.kis_graph()(G, faces=[f for f in G.faces if f.order() == 4])

top_faces = [4]
bottom_faces = []
for f in G.faces:
    if f.order() in top_faces:
        for e in f.halfedge_iter():
            if is_undirected(e):
                e[THIS_WAY] = True
    elif f.order() in bottom_faces:
        for e in f.halfedge_iter():
            if is_undirected(e):
                e.rev[THIS_WAY] = True
right_vertices = filter(lambda v: v.order() == 6, G.vertices)
left_vertices = []
for v in right_vertices:
    for e in v.incoming_iter():
        e[THIS_WAY] = True
for v in left_vertices:
    for e in v.outgoing_iter():
        e[THIS_WAY] = True


# for f in G.faces:
#     if f.order() == 4:
#         for e in f.halfedge_iter():
#             e.rev.nex[THIS_WAY] = True
#             e.rev.pre[THIS_WAY] = True
# for f in G.faces:
#     if f.order() in (6, 12):
#         for e in f.halfedge_iter():
#             if is_undirected(e.rev.nex):
#                 e.rev.nex[THIS_WAY] = True
#             if is_undirected(e.rev.pre):
#                 e.rev.pre[THIS_WAY] = True


# for e in G.border_edges():
#     if e.rev.face in G.faces:
#         G.delete_face(e.rev.face)

# for e in G.border_edges():
#     if e.rev.face.order()==3 and e.nex.rev.face.order() == 3 and e.pre.rev.face.order() == 3:
#         G.delete_face(e.rev.face)
# for e in G.border_edges():
#     if e.rev.face in G.faces and e.rev.nex.rev.on_border():
#         G.delete_face(e.rev.face)
        
G = concentric_rings(12, 2, 2.415)

G = ec.conway.chamfer_graph(0.55)(G, 
                          delete_on_border=True,
                          faces=[f for f in G.faces])# if f.order() == 4])
#G = ec.conway.join_graph()(G, delete_on_border=False,)

ps, vs = G.get_position_view()
#ps[:, 1] *= 1.5
def rot_mat(alpha):
    return np.array([[np.cos(alpha), np.sin(alpha)],[-np.sin(alpha), np.cos(alpha)]])
ps[:] = ps @ rot_mat(np.pi/12)
#ps[:] = [p @ rot_mat(np.linalg.norm(p) / 10) for p in ps]

G.recompute_lengths_and_angles()
G.show(**render_settings)
G.check_consistency()

In [ ]:
n, k = (7, 3)
G = from_tiles(curved_zip(n, k), rings=4)
G.convert_to_euclidean()
G.show(**render_settings)
G.check_consistency()

for f in G.faces:
    if f.order() == 4:
        for e in f.halfedge_iter():
            e[THIS_WAY] = True
for f in G.faces:
    if f.order() == 2*n:
        for e in f.halfedge_iter():
            e.rev[THIS_WAY] = True

In [ ]:
n, k = (7, 3)
print(G.geometry)

#G= ec.conway.dual_graph()(G)

preview_settings = copy(render_settings)
preview_settings['figsize'] = (5, 5)
G = from_tiles(curved_expand(n, k), rings=0)

for i in range(8):
#     print(i)
#     G.show(**preview_settings)
    complete_closest_vertices(G)

    
G.show(**render_settings)
G = ec.conway.kis_graph()(G, faces = [f for f in G.faces if f.order() == n], delete_on_border=True)

G.show(**render_settings)
G.check_consistency()
G.convert_to_euclidean()

for v in G.vertices:
    if v.order() == n:
        for e in v.outgoing_iter():
            e[THIS_WAY] = True
            e.nex[THIS_WAY] = True
            e.nex.rev.pre.rev[THIS_WAY] = True

In [ ]:
def central_face(G):
    fs = list(G.faces)
    return fs[np.argmin([np.linalg.norm(f.midpoint()) for f in fs])]

def central_vertex(G):
    vs = list(G.vertices)
    return vs[np.argmin([np.linalg.norm(v['pos']) for v in vs])]

# G = from_tiles(eu.example_tilesets.t_4_6_12(), rings=3)
G = from_tiles(eu.example_tilesets.platonic(6), rings=6)
# G = from_tiles(eu.example_tilesets.platonic(4), rings=9, vertex_based=False)


# remove the central hexagon
G = conway.kis_graph()(G, faces=[central_face(G)])
v = central_vertex(G)
[G.delete_edge(e.nex) for e in v.outgoing_iter()]
[G.join_vertex(v) for v in list(G.vertices) if v.order() == 2 and not v.on_border()]

#G = conway.dual_graph()(G)
# remove edges crossing the positive x-axis
es = list(G.halfedges)
eps = 1e-6
es = [e for e in es if e.orig['pos'][1] > eps and e.dest['pos'][1] < 0 and e.orig['pos'][0] < 0]
es = sorted(es, key=lambda e: e.orig['pos'][0])
# [G.delete_edge(e) for e in es]

G.show(**render_settings)
ps, vs = G.get_position_view()
#ps[:, 1] *= 1.5
def rot_mat(alpha):
    return np.array([[np.cos(alpha), np.sin(alpha)],[-np.sin(alpha), np.cos(alpha)]])
#ps[:] = ps @ rot_mat(np.pi/12)
k = ps.copy()
k = np.array([complex(*ki) for ki in k])
k -= np.mean(k)

# k = k**1.2 # hex to pentagon
k = k**1.5 # hex to square
# k = k**2 # hex to triangle
# k = k**1.333333333333 # square to triangle

k = np.stack([k.real, k.imag], axis=-1)
ps[:] = k
#ps[:] = [p @ rot_mat(np.linalg.norm(p) / 10) for p in ps]

G.recompute_lengths_and_angles()

from eucare.conversions import EHEG_from_nx

def remove_duplicates(G, eps=1e-6, exclude_edges=()):
    vs = list(G.vertices)
    pos = np.stack([v['pos'] for v in vs])

    dists = np.linalg.norm(pos[:, None] - pos[None, :], axis=-1)
    dists[np.eye(len(vs), dtype=bool)] = np.inf

    closest_points = np.argmin(dists, axis=0)
    min_dists = dists[np.arange(len(pos)), closest_points]
    node_mapping = {i: i if (i<j or d > eps) else j for i, (d, j) in enumerate(zip(min_dists, closest_points))}

    v_index = {v: node_mapping[i] for i, v in enumerate(vs)}

    nxG = nx.Graph()
    nxG.add_nodes_from(v_index.values())
    nx_positions = {i: pos[i] for i in node_mapping.values()}
    nxG.add_edges_from([(v_index[e.orig], v_index[e.dest]) for e in G.halfedges 
                        if e not in set(exclude_edges).union({e.rev for e in exclude_edges})])

    G2 = EHEG_from_nx(nxG, nx_positions)
    G2.recompute_lengths_and_angles()
    return G2

G = remove_duplicates(G, exclude_edges=es)

# G.show(**render_settings)
G = conway.dual_graph()(G)

# G = conway.dual_graph()(G)
G.show(**render_settings)


In [ ]:
# assign creases
# 1. assign each face a value
# use alternating random floodfilling


def bfs_tree(start, neighbor_iter):
    """breathd first search tree starting from a node or set"""
    boundary = start if isinstance(start, set) else set([start])
    parsed = boundary.copy()
    edges = []
    while boundary:
        new_boundary = set()
        for orig in boundary:
            for dest in neighbor_iter(orig):
                if dest not in parsed:
                    parsed.add(dest)
                    new_boundary.add(dest)
                    edges.append((orig, dest))
        boundary = new_boundary
    return edges

def face_bfs_tree(start):
    return bfs_tree(start, lambda f: (f2 for f2 in f.face_iter() if f2 is not None))

def vertex_bfs_tree(start):
    return bfs_tree(start, lambda v: v.vertex_iter())

def assign_this_way_by_z_order(G, key='z_order'):
    for e in G.halfedges:
        if e.on_border() or e.rev.on_border():
            continue
        if e.face[key] > e.rev.face[key]:
            e[THIS_WAY] = True
        else:
            if THIS_WAY in e:
                del e[THIS_WAY] # only in case it was assigned before

In [ ]:
# every second triangle on top, central face on the bottom (works for central face with even order)
cf = central_face(G)
cf['z_order'] = 0
initial_tri = next(f for f in G.faces if f.order() == 3)
initial_tri['z_order'] = 1
for orig, dest in bfs_tree(initial_tri, lambda f: (f2 for f2 in f.face_iter() if f2 is not None and f2.order() == 3)):
    dest['z_order'] = -1 * orig['z_order']
assign_this_way_by_z_order(G)

In [ ]:
# every second triangle on top, central face on the bottom (works for central face with even order)

# get initial faces along one of the four sides
border_edges = [next(h for h in G.border_edge_iter() if h.orig.order() == 3)]
while border_edges[-1].dest.order() == 4:
    border_edges.append(border_edges[-1].nex)
initial_faces = {h.rev.face for h in border_edges}

for f in initial_faces:
    f['z_order'] = 0
for orig, dest in face_bfs_tree(initial_faces):
    dest['z_order'] = orig['z_order'] + 1
    
assign_this_way_by_z_order(G)

In [ ]:
for e in G.halfedges:
    if THIS_WAY in e:
        del e[THIS_WAY]

def pointing_away(e):
    return (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()

vertices = list(G.vertices)
central_vertex = vertices[np.argmin([np.linalg.norm(v['pos']) for v in vertices])]
hops_to_central = {central_vertex: 0}
boundary = {central_vertex}
i = 1
while boundary:
    next_boundary = set()
    for v in boundary:
        for v2 in v.vertex_iter():
            if v2 not in hops_to_central and v2:
                next_boundary.add(v2)
                hops_to_central[v2] = i
    i += 1
    boundary = next_boundary
    
eps = 1e-8
for e in G.halfedges:
    radial_difference = (e.orig['pos']**2).sum() - (e.dest['pos']**2).sum()
    if radial_difference > eps or (abs(radial_difference) < eps and signed_area(np.array([[0, 0], e.orig['pos'], e.dest['pos']])) < 0):
        if hops_to_central[e.orig] % 2 == 0:
            e = e.rev
        e[THIS_WAY] = True
        assert THIS_WAY not in e.rev

In [ ]:
ec.layout.optimize_rotation(G)
SRG = ec.reciprocal_figures.shrink_rotate_graph(G)
SRG.recompute_lengths_and_angles()
assign_shrink_rotate_creases(SRG)

colors = {
    0: (0, 0, 0),
    1: (1, 0, 0),
    -1: (0, 0, 1)
}
for e in SRG.halfedges:
    e['color_key'] = colors[e.attributes.get('crease_assignment', 0)]
    
def simplify_boundary(G):
    # join unneccessary boundary vertices
    to_join = []
    for v in G.vertices:
        if v.on_border() and v.order() == 2:
            to_join.append(v)
    for v in to_join:
        G.join_vertex(v)
    G.recompute_lengths_and_angles()

simplify_boundary(SRG)
SRG.show(**render_settings)

mks = max_kawasaki_sum(SRG)
if mks > 1e-12:
    print('High mks: ', mks)
    
print(f'CP has {len(SRG.halfedges) // 2} edges')

In [ ]:
# some border styles for SRG graphs

class BorderStyle:
    def process_border(G):
        raise NotImplementedError
        
class DefaultBorder(BorderStyle):
    @staticmethod
    def process_border(G):
        return G
    
class DualBorder:
    @staticmethod
    def process_border(G):
        # delete one ring of faces on the border of G
        h = next(h for h in G.halfedges if h.on_border() and 'pre_conway' not in h.rev.face.attributes).rev
        G.delete_subset([v for v in G.vertices if v.on_border()], h.face)
        return G

DualBorder.process_border(SRG)
# simplify_boundary(SRG)
SRG.show()


In [ ]:
from eucare.overlap import line_segment_intersections

class CuttingRegion:
    def inside(self, points):
        raise NotImplementedError
        
    def intersections(self, line_segments):
        raise NotImplementedError
        
    def intersection_order(self, intersections):
        raise NotImplementedError
        
class Halfplane(CuttingRegion):
    def __init__(self, p, v):
        """Halfplane defined by (x - p) * v > 0"""
        self.p = p 
        self.v = v / np.linalg.norm(v)
    
    def signed_distance(self, points):
        return -((points - self.p) * self.v).sum(-1)
    
    def intersections(self, line_segments):
        """
        Compute intersections of the half-plane boundary and the line_segments.
        They are assumed to exist.
        
        The line segments have shape (n_segments, (start, end), (x, y))
        """
        mixing_coefficients = np.abs(((line_segments - self.p) * self.v).sum(-1, keepdims=True))
        mixing_coefficients /= mixing_coefficients.sum(1, keepdims=True)
        mixing_coefficients = 1 - mixing_coefficients
        return (line_segments * mixing_coefficients).sum(1)

def cut_graph(G, region, delete_outside=True, eps=1e-6):
    pts, vs = G.get_position_view()
    dists = region.signed_distance(pts)
    for v, d in zip(vs, dists):
        v['signed_distance'] = d if abs(d) > eps else 0
    
    # find edges that need to be split and split them
    hs = []
    fs = []
    for h in G.halfedges:
        if h.orig['signed_distance'] > 0 and h.dest['signed_distance'] < 0:
            hs.append(h)
        if h.orig['signed_distance'] >= 0 and h.dest['signed_distance'] <= 0:
            fs.extend([h.face, h.rev.face])
    line_segments = np.array([[h.orig['pos'], h.dest['pos']] for h in hs])
    if len(line_segments) == 0:
        return
    intersections = region.intersections(line_segments)
    for h, pos in zip(hs, intersections):
        G.subdivide_edge(h, pos=pos, signed_distance=0)
        
    # add extra edges
    for f in set(fs):
        if f is None:
            continue
        v_inside = set()
        v_outside = set()
        v_border = set()
        for v in f.vertex_iter():
            d = v['signed_distance']
            if d > 0:
                v_outside.add(v)
            elif d < 0:
                v_inside.add(v)
            else:
                v_border.add(v)
        if v_inside and v_outside:
            assert len(v_border) == 2, f'{v_inside}, {v_outside}, {v_border}'
            v1, v2 = list(v_border)
            
            # make it so that the nex path from v1 to v2 is inside
            v1_out = next(h for h in f.halfedge_iter() if h.orig is v1)
            if v1_out.dest['signed_distance'] > 0:
                v1, v2 = v2, v1
                v1_out = next(h for h in f.halfedge_iter() if h.orig is v1)
                
            G.subdivide_face(f, v1, v2)    
            v1_out.pre.rev['new_border'] = True
            
        elif v_outside and len(v_border) > 1:
            assert len(v_border) == 2, f'{v_inside}, {v_border}'
            v1, v2 = list(v_border)
            v1_out = next(h for h in f.halfedge_iter() if h.orig is v1)
            assert v1_out.dest is v2 or v1_out.pre.orig is v2, f'{v1}, {v2}'
            h = v1_out if v1_out.dest is v2 else v1_out.pre
            h['new_border'] = True
            
    if delete_outside:
        # floodfill the outside region and delete it
        to_delete = {h.face for h in G.halfedges if 'new_border' in h.attributes and h.face is not None}
        border = to_delete.copy()
        while border:
            f = border.pop()
            for h in f.halfedge_iter():
                if 'new_border' in h.attributes:
                    continue
                f2 = h.rev.face
                if f2 and f2 not in to_delete:
                    to_delete.add(f2)
                    border.add(f2)
        G.delete_subset(to_delete)
        
    for v in G.vertices:
        del v['signed_distance']
        
def polygon_cut(G, poly, **kwargs):
    if isinstance(poly, ec.half.Face):
        poly =[v['pos'] for v in poly.vertex_iter()]
    poly = np.array(poly)
    assert len(poly.shape) == 2 and poly.shape[-1] == 2, f'{poly.shape}'
    if ec.base.signed_area(poly) < 0:
        poly = poly[::-1]
    rot90 = np.array([[0, -1], [1, 0]])
    halfplanes = [Halfplane(p1, (p1 - p2) @ rot90) 
                  for p1, p2 in zip(poly, ec.half.rotate_by(poly, 1))]
    for hp in halfplanes:
        cut_graph(G, hp, **kwargs)
    return G
        
        
# region = Halfplane(np.array([0.5, 0]), np.array([-1, 0]))
# cut_graph(SRG, region)
SRG.recompute_lengths_and_angles()
h0 = next(h for h in SRG.border_edges())
h = h0
pts = []
while True:
    if h.dest.angle_sum() > np.pi:
        pts.append(h.dest['pos'])
    h = h.nex
    if h is h0:
        break
pts = np.stack(pts)
pts = pts[::-1]
polygon_cut(SRG, pts)
SRG.show(**render_settings)

In [ ]:
%matplotlib notebook
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection
from eucare.plotting import set_equal_aspect
from eucare.reciprocal_figures import random_directed_set

def reshrinkrotate(alpha, factor, global_scale=1):
    for f in faces:
        if 'rotation_center' not in f.attributes:
            continue
        ps, vs = np.array([[v['base_pos'], v] for v in f.vertex_iter()]).T
        ps = np.stack(ps)
        rotation_center = f['rotation_center']

        ps = rotation_center + (ps - rotation_center) @ ec.base.rotation_matrix(alpha) * factor
        
        if global_scale != 1:
            ps *= global_scale
            
        for v, p in zip(vs, ps):
            v['pos'] = p
            
def get_segments(edges):
    return np.array([[e.orig['pos'], e.dest['pos']] for  e in edges])

def get_polys(faces):
    return [[v['pos'] for v in f.vertex_iter()] for f in faces]

edges = list(random_directed_set(SRG.halfedges))
faces = list(SRG.faces)

#%timeit SRG.show(**render_settings)
#%timeit reshrinkrotate(np.pi/9, 0.7)
#%timeit get_segments(edges)
#%timeit get_polys(faces)

segments = get_segments(edges)
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1, 1, 1)
lc = LineCollection(segments, antialiased=True, color='k', linewidth=1)

pc = PolyCollection(get_polys(faces), antialiased=True, color='k')
pc.set_alpha(0.1)

polys = ax.add_collection(pc)
lines = ax.add_collection(lc)

ax.autoscale()
set_equal_aspect()
plt.draw()

alpha_slider = widgets.FloatSlider(0.166666, min=-1, max=1, step=0.02)
factor_slider = widgets.FloatSlider(0.58, min=0, max=6, step=0.05)

last_reparametrized = False
def update(alpha, factor, folded=False, reparametrized=False, scale_folded=False, show_lines=False, show_polys=True):
    alpha = alpha * np.pi
    global last_reparametrized
    if not last_reparametrized:
        gamma = factor / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1)
        beta = np.arccos(np.sin(alpha) / np.sqrt(factor ** 2 - 2 * factor * np.cos(alpha) + 1))
    else:
        gamma = factor
        beta = alpha
        # TODO: sign
        alpha = np.arccos((gamma + np.sin(beta)) / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1))
        factor = gamma / np.sqrt(gamma ** 2 + 2 * gamma * np.sin(beta) + 1)
    
    if reparametrized is not last_reparametrized:
        # adjust slider values
        last_reparametrized = reparametrized
        if reparametrized:
            alpha_slider.value = beta / np.pi
            factor_slider.value = gamma
        else:
            alpha_slider.value = alpha / np.pi
            factor_slider.value = factor
    
    if not folded:
        reshrinkrotate(alpha, factor)
    else:
        factor_folded = gamma / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1)
        alpha_folded = np.sign(alpha) * np.arccos((gamma - np.sin(beta)) / np.sqrt(gamma ** 2 - 2 * gamma * np.sin(beta) + 1))
        reshrinkrotate(alpha_folded, factor_folded, 
                       global_scale=1 if not scale_folded else factor/factor_folded)
    lines.set_segments(get_segments(edges) if show_lines else [])
    polys.set_paths(get_polys(faces) if show_polys else [])
    #ax.draw_artist(lc)
    fig.canvas.draw_idle()
    #print(f'gamma {gamma}, beta {beta * 360 / (2 * np.pi)}')

widgets.interact(
    update,
    alpha=alpha_slider,
    factor=factor_slider,
    
);

In [ ]:
# # for f in SRG.faces:
# #     if 'pre_conway' not in f.attributes:
# #         for h in f.halfedge_iter():
# #             h['in_angle'] *= -1

# ec.overlap.fold_wireframe(SRG)
# # SRG.recompute_positions()
# SRG.show(**render_settings)

4.6.12: 0.166666, 0.789

In [ ]:
from eucare.overlap import fold_complete
SRG.recompute_lengths_and_angles()
result = fold_complete(SRG.copy(), overlap_eps=1e-6, area_eps=0)
render_settings['render_faces'] = False
#result['CP'].show(**render_settings)
#result['folded_view_top'].show(**render_settings)
#result['folded_view_bottom'].show(**render_settings)

In [ ]:
import os
from eucare.redering import SvgwriteRenderer
from eucare.overlap import save_results

path = 'nice_images/cut/test_cut_2' #hex_to_square_6rings_new'
# bbox = (25, 20)
bbox = (35, 30)
# bbox = (65, 48)
#bbox = (95, 58)
save_results(result, path, render_settings, bbox=bbox)

In [ ]:
0.59 * 2/3